# Практика · Пошук і схожість

> Лекція: [lecture.html](lecture.html) · Домашнє завдання: [homework.md](homework.md) ·
> Тест: [quiz.html](quiz.html)

Тут ми рахуємо **всі** числа, які називає лекція, у тому самому порядку.

Що зробимо:

1. Візьмемо ті самі 20 000 документів, що й тема 05, і поставимо задачу пошуку.
2. Побудуємо **інвертований індекс** і заміряємо, скільки він економить.
3. Напишемо **BM25 своїми руками** — бібліотеки `rank_bm25` в системі немає, і це на краще.
4. Розберемо формулу на два механізми: насичення частоти та нормалізацію довжини.
5. Порівняємо TF-IDF і BM25 на 300 запитах × три зерна.
6. Розітнемо виграш BM25 і побачимо, який із двох механізмів його дає.
7. Проженемо сітку `k1` × `b` і подивимось, наскільки результат чутливий.
8. Порівняємо MRR, precision@k і recall@k на трьох запитах, де вони розходяться.
9. Полагодимо одруківки через `rapidfuzz` — і побачимо, де він безсилий.
10. Перевіримо, чи лікує BM25 сліпоту до синонімів. (Спойлер: ні.)

> ⏱ Заміряно на чотирьох ядрах без відеокарти: **від хвилини до двох з
> половиною**, залежно від завантаження машини. Три прогони поспіль дали 66, 80
> і 151 секунду — останній ішов, коли поруч рахувало ще щось. Найдовше йде
> сітка `k1` × `b` (30 положень × три зерна) і повний перебір без індексу.

## 1 · Середовище

Перша клітинка друкує версії. Якщо в тебе інші — числа можуть трохи поїхати,
і краще знати про це одразу, а не наприкінці.

In [ ]:
import sys, re, math, glob, gettext, time, collections
import numpy as np
import scipy.sparse as sp
import sklearn
import rapidfuzz
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize

print("Python   ", sys.version.split()[0])
print("numpy    ", np.__version__)
print("scipy    ", __import__("scipy").__version__)
print("sklearn  ", sklearn.__version__)
print("rapidfuzz", rapidfuzz.__version__)

# rank_bm25 в системі немає — і саме тому формулу ми напишемо самі
try:
    import rank_bm25
    print("rank_bm25", rank_bm25.__version__)
except ImportError:
    print("rank_bm25 немає — пишемо BM25 своїми руками")

## 2 · Корпус: ті самі переклади інтерфейсів

Корпус блоку не міняється від теми до теми: українські переклади в системних
файлах `.mo`. Кожен запис — пара «англійський оригінал → український переклад»;
документом ми вважаємо **переклад**.

Якщо української локалі на машині немає, вмикається вбудований мінікорпус —
чисел він, звісно, не відтворить, але зошит виконається до кінця.

In [ ]:
MINI = [
    ("mini", "File not found", "Файл «%s» не знайдено на носії «%s»"),
    ("mini", "Cannot open file", "Не вдалося відкрити файл налаштувань програми"),
    ("mini", "Timeout", "Перевищено час очікування відповіді від сервера"),
    ("mini", "Invalid certificate", "Сертифікат сервера недійсний або прострочений"),
    ("mini", "Permission denied", "Відмовлено у доступі до каталогу користувача"),
    ("mini", "Disk read error", "Помилка читання диска у вказаному розділі"),
    ("mini", "Screen resolution", "Роздільність екрана не підтримується драйвером"),
    ("mini", "Move window", "Перемістити вікно на останній робочий простір"),
] * 40


def load_corpus():
    """Пари «оригінал → переклад» з усіх .mo українською локаллю."""
    documents = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as handle:
                catalog = gettext.GNUTranslations(handle)
            program = path.split('/')[-1][:-3]
            for source, target in catalog._catalog.items():
                if isinstance(source, str) and isinstance(target, str) \
                   and len(target) > 30 and 'Project-Id' not in target:
                    documents.append((program, source, target))
        except Exception:
            pass
    return documents


corpus = load_corpus()
if len(corpus) < 5000:
    print("української локалі мало або немає — вмикаю вбудований мінікорпус")
    corpus = MINI
    source_label = "мінікорпус"
else:
    source_label = "/usr/share/locale/uk/LC_MESSAGES/*.mo"

programs = {program for program, _, _ in corpus}
print("джерело     :", source_label)
print("документів  :", len(corpus))
print("програм     :", len(programs))
print("приклад     :", corpus[17][2])

## 3 · Токенізатор і 20 000 документів у трьох примірниках

Токенізатор блоку — той самий рядок, що в темах 01-05. Брати свій не можна:
у блоці 1 розбіжність на пів відсотка мало не розсипала числа, які теми
цитують одна в одної.

Далі беремо **20 000 випадкових документів** і робимо це **тричі**, із зернами
0, 1 і 2. Різниця між двома способами пошуку має сенс лише тоді, коли вона
більша за розкид між зернами.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"    # після lowercase=True
tokenize = re.compile(TOKEN_PATTERN).findall

SEEDS = (0, 1, 2)
N_DOCS = 20000

documents = [target for _, _, target in corpus]


def sample_documents(seed, n=N_DOCS):
    """Ті самі 20 000 документів при тому самому зерні — на будь-якій машині."""
    rng = np.random.default_rng(seed)
    picked = rng.choice(len(documents), min(n, len(documents)), replace=False)
    return [documents[i] for i in picked]


def count_matrix(texts):
    """Мішок слів із теми 04: рядок — документ, колонка — слово, клітинка — скільки разів."""
    vectorizer = CountVectorizer(token_pattern=TOKEN_PATTERN)
    counts = vectorizer.fit_transform(texts).tocsr()
    return vectorizer, counts


# матриці рахуємо один раз і тримаємо: далі вони знадобляться разів десять
SAMPLES = {}
for seed in SEEDS:
    texts = sample_documents(seed)
    vectorizer, counts = count_matrix(texts)
    SAMPLES[seed] = (texts, vectorizer, counts)
    total_length = np.asarray(counts.sum(axis=1)).ravel()
    unique_length = np.diff(counts.indptr)
    print(f"зерно {seed}: {counts.shape[0]} документів, {counts.shape[1]} слів у словнику, "
          f"довжина документа {total_length.mean():.4f} слововживань "
          f"({unique_length.mean():.4f} різних слів)")

## 4 · Задача: знайти документ за трьома його словами

Ця задача називається **пошуком відомого документа**: ми беремо документ,
робимо з нього запит із трьох випадкових слів і дивимось, на якому місці
пошук поверне саме його. Тема 05 міряла на ній TF-IDF; ми беремо той самий
код дослівно, щоб числа двох тем можна було ставити поруч.

Метрика — **MRR** (mean reciprocal rank, середній обернений ранг). Якщо
потрібний документ виявився першим, запит дає 1; другим — 1/2; десятим — 1/10.
MRR — це середнє таких дробів по всіх запитах.

In [ ]:
def build_queries(counts, seed, n_queries=300, mode="random"):
    """З кожного документа-мішені робимо запит із трьох його слів.
    mode='random' — три випадкові слова документа;
    mode='mixed'  — два найчастіші в корпусі слова документа плюс одне найрідкісніше."""
    rng = np.random.default_rng(1000 + seed)
    df = np.asarray((counts > 0).sum(axis=0)).ravel()
    lengths = np.diff(counts.indptr)
    targets = rng.choice(np.where(lengths >= 6)[0], n_queries, replace=False)
    rows, columns = [], []
    for number, target in enumerate(targets):
        start, stop = counts.indptr[target], counts.indptr[target + 1]
        terms = counts.indices[start:stop]
        if mode == "mixed":
            by_df = terms[np.argsort(-df[terms])]
            picked = [by_df[0], by_df[1], by_df[-1]]
        else:
            picked = rng.choice(terms, 3, replace=False)
        for term in picked:
            rows.append(number)
            columns.append(term)
    queries = sp.csr_matrix((np.ones(len(rows)), (rows, columns)),
                            shape=(n_queries, counts.shape[1]))
    return queries, targets


def places_of_targets(scores, targets, n_documents):
    """Місце мішені у видачі. Нічиї ріжемо за номером документа —
    інакше однакові рядки корпусу дали б різний ранг у різних прогонах."""
    own = scores[np.arange(len(targets)), targets]
    better = (scores > own[:, None]).sum(axis=1)
    ties_before = ((scores == own[:, None]) &
                   (np.arange(n_documents)[None, :] < targets[:, None])).sum(axis=1)
    return 1 + better + ties_before


def mrr_of(scores, targets):
    places = places_of_targets(scores, targets, scores.shape[1])
    return float(np.mean(1.0 / places)), places


QUERIES = {}
for seed in SEEDS:
    texts, vectorizer, counts = SAMPLES[seed]
    QUERIES[seed] = build_queries(counts, seed)

texts, vectorizer, counts = SAMPLES[0]
words = list(vectorizer.get_feature_names_out())
queries, targets = QUERIES[0]
for number in (0, 1, 2):
    terms = queries.indices[queries.indptr[number]:queries.indptr[number + 1]]
    print("запит :", " ".join(words[t] for t in terms))
    print("мішень:", texts[targets[number]][:70])

## 5 · Інвертований індекс

Перш ніж рахувати якість, треба навчитися рахувати **швидко**. Наївний спосіб
відповісти на запит — пройти всі 20 000 документів і кожному нарахувати оцінку.
Інвертований індекс робить інакше: для кожного слова наперед зберігає список
документів, у яких воно є, і на запит обходить **лише об'єднання цих списків**.

Заміряємо обидва способи чесно: обидва написані звичайним Python, обидва
рахують ту саму суму ваг.

In [ ]:
def bm25_idf(counts):
    """IDF у формі BM25: ln(1 + (N − df + 0.5) / (df + 0.5))."""
    n_documents = counts.shape[0]
    df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
    return np.log(1.0 + (n_documents - df + 0.5) / (df + 0.5))


def bm25_matrix(counts, k1=1.5, b=0.75):
    """Вага кожної клітинки за BM25. Від запиту вона не залежить,
    тож рахуємо її один раз на колекцію, а не на кожен запит."""
    lengths = np.asarray(counts.sum(axis=1)).ravel().astype(float)
    average_length = lengths.mean()
    idf = bm25_idf(counts)
    weighted = counts.tocsr().astype(float).copy()
    row_of_cell = np.repeat(np.arange(weighted.shape[0]), np.diff(weighted.indptr))
    length_part = k1 * (1 - b + b * lengths[row_of_cell] / average_length)
    tf = weighted.data
    weighted.data = idf[weighted.indices] * tf * (k1 + 1) / (tf + length_part)
    return weighted


texts, vectorizer, counts = SAMPLES[0]
weights = bm25_matrix(counts)
queries, targets = QUERIES[0]
query_terms = [list(queries.indices[queries.indptr[i]:queries.indptr[i + 1]])
               for i in range(queries.shape[0])][:60]

# спосіб 1: словник на кожен документ і повний обхід усіх документів
per_document = [dict(zip(weights.indices[weights.indptr[i]:weights.indptr[i + 1]],
                         weights.data[weights.indptr[i]:weights.indptr[i + 1]]))
                for i in range(weights.shape[0])]

# спосіб 2: інвертований індекс — для кожного слова список (документ, вага)
by_column = weights.tocsc()
postings = {term: list(zip(by_column.indices[by_column.indptr[term]:by_column.indptr[term + 1]],
                           by_column.data[by_column.indptr[term]:by_column.indptr[term + 1]]))
            for term in range(weights.shape[1])}

start = time.perf_counter()
for terms in query_terms:
    best_score, best_document = -1.0, -1
    for number, document in enumerate(per_document):
        score = 0.0
        for term in terms:
            weight = document.get(term)
            if weight is not None:
                score += weight
        if score > best_score:
            best_score, best_document = score, number
scan_time = (time.perf_counter() - start) / len(query_terms) * 1000

start = time.perf_counter()
touched = []
for terms in query_terms:
    accumulator = collections.defaultdict(float)
    for term in terms:
        for number, weight in postings[term]:
            accumulator[number] += weight
    touched.append(len(accumulator))
    best_document = max(accumulator, key=accumulator.get) if accumulator else -1
index_time = (time.perf_counter() - start) / len(query_terms) * 1000

print(f"повний перебір     : {scan_time:.1f} мс на запит")
print(f"з індексом       : {index_time:.1f} мс на запит")
print(f"прискорення        : {scan_time / index_time:.1f} разів")
print("час залежить від завантаження машини; стабільне тут — не мілісекунди, "
      "а скільки документів довелось обійти")
print(f"торкнулись документів: медіана {np.median(touched):.0f} із {counts.shape[0]}, "
      f"середнє {np.mean(touched):.1f}")

Тепер найважливіше про індекс: **скільки він економить, залежить від запиту**,
а не від розміру колекції. Візьмемо чотири справжні запити різної рідкісності
й подивимось, скільки документів доведеться торкнутись у кожному випадку.

In [ ]:
DEMO_QUERIES = [
    ["не", "вдалося", "файл"],
    ["помилка", "читання", "диска"],
    ["сертифікат", "недійсний"],
    ["роздільність", "екрана"],
]


def round_half_up(value, digits=2):
    """Python округлює половини до парного, а JavaScript — угору.
    Ці ж числа малює канва лекції, тож рахуємо так, як рахує вона."""
    factor = 10 ** digits
    return math.floor(value * factor + 0.5) / factor


for seed in SEEDS:
    texts, vectorizer, counts = SAMPLES[seed]
    vocabulary = vectorizer.vocabulary_
    by_column = (counts > 0).tocsc()
    print(f"зерно {seed}:")
    for query_words in DEMO_QUERIES:
        columns = [vocabulary[word] for word in query_words if word in vocabulary]
        document_frequencies = [int(by_column.indptr[c + 1] - by_column.indptr[c]) for c in columns]
        union = set()
        for c in columns:
            union.update(by_column.indices[by_column.indptr[c]:by_column.indptr[c + 1]])
        share = round_half_up(100 * len(union) / counts.shape[0])
        print(f"   {' '.join(query_words):<26} df {document_frequencies}  "
              f"торкнулись {len(union):>5} ({share:6.2f} %)  "
              f"економія {counts.shape[0] / max(len(union), 1):7.1f}x")

І третій бік того самого: чому індекс — це не лише про час, а й про памʼять.
Та сама матриця ваг у щільному вигляді та в розрідженому.

In [ ]:
texts, vectorizer, counts = SAMPLES[0]
weights = bm25_matrix(counts)
dense_bytes = weights.shape[0] * weights.shape[1] * 8          # float64
sparse_bytes = weights.data.nbytes + weights.indices.nbytes + weights.indptr.nbytes
print(f"щільна матриця   : {dense_bytes / 1e6:10.2f} МБ "
      f"({weights.shape[0]} x {weights.shape[1]} клітинок)")
print(f"розріджена (CSR) : {sparse_bytes / 1e6:10.2f} МБ "
      f"(data {weights.data.nbytes / 1e6:.2f} + indices {weights.indices.nbytes / 1e6:.2f} "
      f"+ indptr {weights.indptr.nbytes / 1e6:.2f})")
print(f"різниця          : {dense_bytes / sparse_bytes:10.0f} разів")

## 6 · Базова лінія: косинус TF-IDF

Тема 05 закінчила на тому, що документ — це вектор ваг TF-IDF, а схожість —
косинус кута між векторами. Перевіримо спершу, що наш косинус збігається з
бібліотечним, і лише потім будемо його з чимось порівнювати.

In [ ]:
def tfidf_idf(counts):
    """IDF у формі sklearn: ln((1 + N) / (1 + df)) + 1."""
    n_documents = counts.shape[0]
    df = np.asarray((counts > 0).sum(axis=0)).ravel().astype(float)
    return np.log((1 + n_documents) / (1 + df)) + 1.0


def tfidf_scores(counts, queries):
    """Косинус TF-IDF: обидві сторони зважені й L2-нормовані."""
    idf = tfidf_idf(counts)
    document_side = normalize(counts.multiply(idf).tocsr())
    query_side = normalize(queries.multiply(idf).tocsr())
    return (query_side @ document_side.T).toarray()


# звірка з бібліотекою: та сама матриця, порахована TfidfVectorizer
texts, vectorizer, counts = SAMPLES[0]
library = TfidfVectorizer(token_pattern=TOKEN_PATTERN)
library_matrix = library.fit_transform(texts[:2000])
_, small_counts = count_matrix(texts[:2000])
ours = normalize(small_counts.multiply(tfidf_idf(small_counts)).tocsr())
assert np.allclose(ours.toarray(), library_matrix.toarray()), "TF-IDF розійшовся з sklearn!"
print("✅ наш TF-IDF збігається з TfidfVectorizer до останнього знака")

tfidf_runs = []
for seed in SEEDS:
    texts, vectorizer, counts = SAMPLES[seed]
    queries, targets = QUERIES[seed]
    value, _ = mrr_of(tfidf_scores(counts, queries), targets)
    tfidf_runs.append(value)
    print(f"зерно {seed}: TF-IDF MRR {value:.4f}")
print(f"середнє {np.mean(tfidf_runs):.4f}, розкид {max(tfidf_runs) - min(tfidf_runs):.4f}")

## 7 · BM25 своїми руками

Формула виглядає так:

    score(q, d) = Σ  idf(t) · ( tf(t,d) · (k1 + 1) ) / ( tf(t,d) + k1 · (1 − b + b · |d| / avgdl) )
                 t∈q

Розберемо її на людській мові:

* `tf(t,d)` — скільки разів слово `t` трапилось у документі `d`;
* `|d|` — довжина документа в словах, `avgdl` — середня довжина документа колекції;
* `k1` керує **насиченням**: наскільки швидко перестає рости внесок повторів;
* `b` керує **нормалізацією довжини**: наскільки сильно карати довгий документ;
* `idf(t)` — та сама ідея, що в темі 05, тільки в іншій формі.

Спершу порахуємо це руками на одному документі — щоб було видно, що чисел там
рівно стільки, скільки в рядку формули.

In [ ]:
def bm25_by_hand(tf, document_length, average_length, idf, k1=1.5, b=0.75):
    """Один доданок формули, розписаний по кроках."""
    length_penalty = 1 - b + b * document_length / average_length
    denominator = tf + k1 * length_penalty
    numerator = tf * (k1 + 1)
    return idf * numerator / denominator, length_penalty, numerator, denominator


texts, vectorizer, counts = SAMPLES[0]
vocabulary = vectorizer.vocabulary_
lengths = np.asarray(counts.sum(axis=1)).ravel().astype(float)
average_length = lengths.mean()
idf_values = bm25_idf(counts)

# документ шукаємо за текстом, а не за номером: номер залежить від корпусу
document_number = next((i for i, t in enumerate(texts)
                        if t.startswith("Файл «%s» не знайдено")), 0)
print("документ:", repr(texts[document_number]))
print(f"довжина {lengths[document_number]:.0f} слів, середня довжина колекції {average_length:.4f}")
print()
total = 0.0
for word in [w for w in ["файл", "не", "знайдено"] if w in vocabulary]:
    column = vocabulary[word]
    tf = float(counts[document_number, column])
    part, penalty, numerator, denominator = bm25_by_hand(
        tf, lengths[document_number], average_length, idf_values[column])
    total += part
    print(f"  {word:<10} tf {tf:.0f}  idf {idf_values[column]:.4f}  "
          f"штраф за довжину {penalty:.4f}  {numerator:.4f}/{denominator:.4f}  внесок {part:.4f}")
print(f"  разом: {total:.4f}")

Тепер те саме, але для всієї колекції одразу — і **обовʼязкова звірка**:
швидка векторна версія мусить дати те саме, що повільний потрійний цикл.
`rank_bm25` у системі немає, тож еталоном тут є наша ж наївна реалізація,
написана буквально по рядку формули.

In [ ]:
def bm25_slowly(counts, query_columns, k1=1.5, b=0.75):
    """Та сама формула, але циклами: рядок за рядком, слово за словом."""
    lengths = np.asarray(counts.sum(axis=1)).ravel().astype(float)
    average_length = lengths.mean()
    idf = bm25_idf(counts)
    scores = np.zeros(counts.shape[0])
    for document in range(counts.shape[0]):
        start, stop = counts.indptr[document], counts.indptr[document + 1]
        present = dict(zip(counts.indices[start:stop], counts.data[start:stop]))
        for column in query_columns:
            tf = float(present.get(column, 0))
            if tf == 0:
                continue
            penalty = 1 - b + b * lengths[document] / average_length
            scores[document] += idf[column] * tf * (k1 + 1) / (tf + k1 * penalty)
    return scores


def bm25_scores(counts, queries, k1=1.5, b=0.75):
    """Векторна версія: множимо матрицю ваг на двійковий вектор запиту."""
    weights = bm25_matrix(counts, k1, b)
    query_side = (queries > 0).astype(float)
    return (query_side @ weights.T).toarray()


texts, vectorizer, counts = SAMPLES[0]
small_texts = texts[:3000]
_, small_counts = count_matrix(small_texts)
small_vocabulary = count_matrix(small_texts)[0].vocabulary_
check_columns = [small_vocabulary[w] for w in ["файл", "не", "знайдено"] if w in small_vocabulary]
one_query = sp.csr_matrix((np.ones(len(check_columns)),
                           (np.zeros(len(check_columns), dtype=int), check_columns)),
                          shape=(1, small_counts.shape[1]))
fast = bm25_scores(small_counts, one_query)[0]
slow = bm25_slowly(small_counts, check_columns)
assert np.allclose(fast, slow), "векторна BM25 розійшлася з наївною!"
print(f"✅ векторна BM25 = наївна BM25 на {small_counts.shape[0]} документах, "
      f"максимальна розбіжність {np.abs(fast - slow).max():.2e}")

## 8 · Механізм перший: насичення частоти

TF-IDF вірить, що двадцяте входження слова важить рівно у двадцять разів
більше за перше. BM25 у це не вірить. Порахуємо внесок частоти в обох схемах
на документі середньої довжини (щоб множник довжини дорівнював одиниці).

In [ ]:
K1 = 1.5
print(f"внесок частоти при k1 = {K1}, документ середньої довжини:")
print(f"{'входжень':>10} {'TF-IDF':>10} {'BM25':>10} {'BM25/попереднє':>16}")
previous = None
saturation_table = []
for tf in (1, 2, 3, 5, 10, 20):
    bm25_part = tf * (K1 + 1) / (tf + K1)
    saturation_table.append((tf, bm25_part))
    step = "" if previous is None else f"{bm25_part - previous:+.4f}"
    print(f"{tf:>10} {tf:>10} {bm25_part:>10.4f} {step:>16}")
    previous = bm25_part
ceiling = K1 + 1
print(f"\nстеля внеску при tf -> нескінченність: k1 + 1 = {ceiling:.4f}")
print(f"двадцяте входження додає {saturation_table[-1][1] - (19 * (K1+1)/(19+K1)):.4f}, "
      f"друге додало {saturation_table[1][1] - saturation_table[0][1]:.4f}")

І одразу перевірка, чи має це насичення на нашому корпусі про що говорити:
скільки взагалі в ньому клітинок із `tf > 1`.

In [ ]:
for seed in SEEDS:
    texts, vectorizer, counts = SAMPLES[seed]
    repeated_cells = 100 * np.mean(counts.data > 1)
    has_repeat = np.array([(counts.data[counts.indptr[i]:counts.indptr[i + 1]] > 1).any()
                           for i in range(counts.shape[0])])
    print(f"зерно {seed}: клітинок із tf>1 — {repeated_cells:.2f} %, "
          f"документів хоча б з одним повтореним словом — {100 * has_repeat.mean():.2f} %")

## 9 · Механізм другий: нормалізація довжини через `b`

Тепер найнаочніша частина теми. Візьмемо справжній запит **«файл не знайдено»**
і подивимось, який документ виявляється першим при різних `b`.

In [ ]:
texts, vectorizer, counts = SAMPLES[0]
vocabulary = vectorizer.vocabulary_
query_words = [w for w in ["файл", "не", "знайдено"] if w in vocabulary]
columns = [vocabulary[w] for w in query_words]
one_query = sp.csr_matrix((np.ones(len(columns)),
                           (np.zeros(len(columns), dtype=int), columns)),
                          shape=(1, counts.shape[1]))
lengths = np.asarray(counts.sum(axis=1)).ravel()

print("idf слів запиту:",
      {w: round(float(bm25_idf(counts)[vocabulary[w]]), 4) for w in query_words})
print(f"середня довжина документа: {lengths.mean():.4f}\n")
for b in (0.0, 0.1, 0.15, 0.25, 0.5, 0.75, 1.0):
    scores = bm25_scores(counts, one_query, k1=1.5, b=b)[0]
    winner = int(np.argmax(scores))
    tf_here = [int(counts[winner, c]) for c in columns]
    print(f"b = {b:.2f}  топ-1 має {lengths[winner]:>3} слів, tf {tf_here}, "
          f"оцінка {scores[winner]:6.4f}")
    print(f"           {texts[winner][:76]!r}")

Один документ — це ще не замір. Проженемо той самий `b` по всіх 300 запитах
і трьох зернах: як міняється якість і **яку довжину має документ, що став
першим**.

In [ ]:
length_table = []
for b in (0.0, 0.25, 0.5, 0.75, 1.0):
    quality, top_length = [], []
    for seed in SEEDS:
        texts, vectorizer, counts = SAMPLES[seed]
        queries, targets = QUERIES[seed]
        scores = bm25_scores(counts, queries, k1=1.5, b=b)
        value, _ = mrr_of(scores, targets)
        quality.append(value)
        lengths = np.asarray(counts.sum(axis=1)).ravel()
        top_length.append(float(lengths[np.argmax(scores, axis=1)].mean()))
    length_table.append((b, np.mean(quality), np.mean(top_length)))
    print(f"b = {b:.2f}  MRR {np.mean(quality):.4f} (розкид {max(quality)-min(quality):.4f})  "
          f"середня довжина топ-1 {np.mean(top_length):.2f} слова")
texts, vectorizer, counts = SAMPLES[0]
print(f"для порівняння: середня довжина документа колекції "
      f"{np.asarray(counts.sum(axis=1)).ravel().mean():.2f} слова")

## 10 · Головний замір теми: TF-IDF проти BM25

Триста запитів, три зерна, та сама колекція, та сама метрика.

In [ ]:
main_table = {}
for name, scorer in [("TF-IDF косинус", lambda c, q: tfidf_scores(c, q)),
                     ("BM25 k1=1.5 b=0.75", lambda c, q: bm25_scores(c, q, 1.5, 0.75))]:
    runs = []
    for seed in SEEDS:
        texts, vectorizer, counts = SAMPLES[seed]
        queries, targets = QUERIES[seed]
        value, _ = mrr_of(scorer(counts, queries), targets)
        runs.append(value)
    main_table[name] = runs
    print(f"{name:<20} " + "  ".join(f"зерно {s}: {v:.4f}" for s, v in zip(SEEDS, runs)))
    print(f"{'':<20} середнє {np.mean(runs):.4f}, розкид {max(runs) - min(runs):.4f}")

gains = [b - a for a, b in zip(main_table["TF-IDF косинус"], main_table["BM25 k1=1.5 b=0.75"])]
print()
print("виграш BM25 по зернах: " + ", ".join(f"{g:+.4f}" for g in gains))
print(f"середній виграш {np.mean(gains):+.4f}, розкид виграшу {max(gains) - min(gains):.4f}")
print(f"найкраще зерно TF-IDF {max(main_table['TF-IDF косинус']):.4f} "
      f"проти найгіршого зерна BM25 {min(main_table['BM25 k1=1.5 b=0.75']):.4f} — "
      f"{'купи не перетинаються' if max(main_table['TF-IDF косинус']) < min(main_table['BM25 k1=1.5 b=0.75']) else 'купи перетинаються'}")

Той самий замір на другому типі запитів — «два звичні слова плюс одне
змістовне». Тема 05 називала його реалістичним; треба переконатися, що
висновок не тримається на одному способі складати запити.

In [ ]:
mixed_queries = {seed: build_queries(SAMPLES[seed][2], seed, mode="mixed") for seed in SEEDS}
for name, scorer in [("TF-IDF косинус", lambda c, q: tfidf_scores(c, q)),
                     ("BM25 k1=1.5 b=0.75", lambda c, q: bm25_scores(c, q, 1.5, 0.75))]:
    runs = []
    for seed in SEEDS:
        texts, vectorizer, counts = SAMPLES[seed]
        queries, targets = mixed_queries[seed]
        value, _ = mrr_of(scorer(counts, queries), targets)
        runs.append(value)
    print(f"запити «mixed», {name:<20} MRR {np.mean(runs):.4f}, "
          f"розкид {max(runs) - min(runs):.4f}")

## 11 · Розтин: звідки береться виграш

Різницю в 0.08 MRR дають два механізми — насичення та нормалізація довжини.
Але ще BM25 користується **іншою формулою IDF**. Вимкнемо по одному й
подивимось, що саме працює.

In [ ]:
def cosine_with(counts, queries, idf, binary=False):
    """L2-косинус із довільними вагами слів — щоб міняти по одному складнику."""
    document_side = (counts > 0).astype(float) if binary else counts.astype(float)
    query_side = (queries > 0).astype(float)
    return (normalize(query_side.multiply(idf).tocsr())
            @ normalize(document_side.multiply(idf).tocsr()).T).toarray()


ablation = [
    ("1 TF-IDF, L2-косинус (тема 05)", lambda c, q: cosine_with(c, q, tfidf_idf(c))),
    ("2 те саме, але IDF за формулою BM25", lambda c, q: cosine_with(c, q, bm25_idf(c))),
    ("3 BM25 IDF, двійковий tf, L2-косинус", lambda c, q: cosine_with(c, q, bm25_idf(c), binary=True)),
    ("4 BM25 повний, k1=1.5 b=0.75", lambda c, q: bm25_scores(c, q, 1.5, 0.75)),
    ("5 BM25 без нормалізації довжини, b=0", lambda c, q: bm25_scores(c, q, 1.5, 0.0)),
    ("6 BM25 без насичення, k1=1000", lambda c, q: bm25_scores(c, q, 1000.0, 0.75)),
    ("7 BM25 з повним насиченням, k1=0", lambda c, q: bm25_scores(c, q, 0.0, 0.75)),
]
ablation_runs = {}
for name, scorer in ablation:
    runs = []
    for seed in SEEDS:
        texts, vectorizer, counts = SAMPLES[seed]
        queries, targets = QUERIES[seed]
        value, _ = mrr_of(scorer(counts, queries), targets)
        runs.append(value)
    ablation_runs[name] = runs
    print(f"{name:<40} MRR {np.mean(runs):.4f}  розкид {max(runs) - min(runs):.4f}")
full = np.mean(ablation_runs["4 BM25 повний, k1=1.5 b=0.75"])
print()
print(f"вимкнути насичення коштує {np.mean(ablation_runs['6 BM25 без насичення, k1=1000']) - full:+.4f}")
print(f"вимкнути нормалізацію довжини коштує "
      f"{np.mean(ablation_runs['5 BM25 без нормалізації довжини, b=0']) - full:+.4f}")
print(f"сама лише зміна формули IDF дає "
      f"{np.mean(ablation_runs['2 те саме, але IDF за формулою BM25']) - np.mean(ablation_runs['1 TF-IDF, L2-косинус (тема 05)']):+.4f}")

## 12 · Сітка `k1` × `b`

Значення `k1 = 1.5`, `b = 0.75` цитують як «типові». Перевіримо їх на нашій
колекції: шість значень `k1` на пʼять значень `b`, кожна клітинка — три зерна.

In [ ]:
K1_GRID = (0.0, 0.3, 0.6, 1.2, 1.5, 3.0)
B_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)

grid = {}
for k1 in K1_GRID:
    for b in B_GRID:
        runs = []
        for seed in SEEDS:
            texts, vectorizer, counts = SAMPLES[seed]
            queries, targets = QUERIES[seed]
            value, _ = mrr_of(bm25_scores(counts, queries, k1, b), targets)
            runs.append(value)
        grid[(k1, b)] = (float(np.mean(runs)), float(max(runs) - min(runs)))

print("MRR: рядки — k1, колонки — b")
print(f"{'k1 \\ b':>8}" + "".join(f"{b:>9.2f}" for b in B_GRID))
for k1 in K1_GRID:
    print(f"{k1:>8.1f}" + "".join(f"{grid[(k1, b)][0]:>9.4f}" for b in B_GRID))

best_cell = max(grid, key=lambda key: grid[key][0])
default_value, default_spread = grid[(1.5, 0.75)]
best_value, best_spread = grid[best_cell]
inside = [cell for cell in grid if grid[cell][0] >= best_value - default_spread]
print()
print(f"типові значення k1=1.5 b=0.75 : MRR {default_value:.4f}, розкид {default_spread:.4f}")
print(f"найкраща клітинка k1={best_cell[0]} b={best_cell[1]} : MRR {best_value:.4f}, "
      f"розкид {best_spread:.4f}")
print(f"перевага найкращої над типовою: {best_value - default_value:+.4f}")
print(f"клітинок, що не гірші за найкращу в межах розкиду: {len(inside)} із {len(grid)}")
print("це:", ", ".join(f"k1={c[0]} b={c[1]}" for c in sorted(inside)))
print()
print("рядок k1=0 однаковий по всіх b — і це не помилка: при k1=0 множник довжини "
      "входить у формулу лише через k1, тож b перестає щось означати, а сама BM25 "
      "перетворюється на просту суму IDF наявних слів запиту.")

## 13 · Чим міряти пошук: MRR, precision@k, recall@k

Досі ми міряли одним числом. Тепер поставимо поруч три і покажемо запити, на
яких вони кажуть протилежне.

Розмітка доречності тут перевіряється очима: **доречний документ — той, у якому
є всі слова запиту**. Це грубо, але чесно й відтворювано.

In [ ]:
def relevance_mask(counts, queries):
    """Доречний = містить усі слова запиту."""
    binary_documents = (counts > 0).astype(np.int8)
    binary_queries = (queries > 0).astype(np.int8)
    hits = (binary_queries @ binary_documents.T).toarray()
    needed = np.asarray(binary_queries.sum(axis=1)).ravel()
    return hits == needed[:, None]


def three_metrics(scores, relevant, k=10):
    order = np.argsort(-scores, axis=1)
    marks = np.take_along_axis(relevant, order, axis=1)
    precision = marks[:, :k].mean(axis=1)
    totals = relevant.sum(axis=1)
    recall = marks[:, :k].sum(axis=1) / np.maximum(totals, 1)
    first_relevant = np.argmax(marks, axis=1) + 1
    return precision, recall, 1.0 / first_relevant, totals


for seed in SEEDS:
    texts, vectorizer, counts = SAMPLES[seed]
    queries, targets = QUERIES[seed]
    relevant = relevance_mask(counts, queries)
    totals = relevant.sum(axis=1)
    print(f"зерно {seed}: доречних на запит — медіана {np.median(totals):.0f}, "
          f"середнє {totals.mean():.2f}, максимум {totals.max()}; "
          f"запитів рівно з одним доречним {100 * np.mean(totals == 1):.1f} %")
    for name, scores in [("TF-IDF", tfidf_scores(counts, queries)),
                         ("BM25", bm25_scores(counts, queries, 1.5, 0.75))]:
        precision, recall, reciprocal, _ = three_metrics(scores, relevant)
        print(f"     {name:<8} MRR {reciprocal.mean():.4f}  "
              f"precision@10 {precision.mean():.4f}  recall@10 {recall.mean():.4f}")

Середні числа приховують найцікавіше. Подивимось на три конкретні запити, у
яких доречних документів різна кількість: один, тринадцять і сто двадцять.

In [ ]:
texts, vectorizer, counts = SAMPLES[0]
vocabulary = vectorizer.vocabulary_
relevance_all = (counts > 0)


def show_query(query_words, k=10, known_item=None):
    query_words = [w for w in query_words if w in vocabulary]
    columns = [vocabulary[w] for w in query_words]
    one = sp.csr_matrix((np.ones(len(columns)), (np.zeros(len(columns), dtype=int), columns)),
                        shape=(1, counts.shape[1]))
    scores = bm25_scores(counts, one, 1.5, 0.75)[0]
    if known_item is None:
        relevant = np.asarray(relevance_all[:, columns].sum(axis=1)).ravel() == len(columns)
        label = "доречний = містить усі слова запиту"
    else:
        relevant = np.zeros(counts.shape[0], dtype=bool)
        relevant[known_item] = True
        label = "доречний = один конкретний документ"
    order = np.argsort(-scores)[:k]
    print(f"ЗАПИТ «{' '.join(query_words)}» — {label}")
    print(f"   доречних у колекції: {int(relevant.sum())}")
    for place, document in enumerate(order, 1):
        mark = "+" if relevant[document] else " "
        print(f"   {place:>2} {mark} {scores[document]:7.4f}  {texts[document][:64]}")
    first = int(np.argmax(relevant[np.argsort(-scores)])) + 1
    hits = relevant[order].sum()
    print(f"   MRR = 1/{first} = {1 / first:.4f}   precision@10 = {hits / k:.4f}   "
          f"recall@10 = {hits / relevant.sum():.4f}")
    print()


known_target = next((i for i, t in enumerate(texts)
                     if t.startswith("Визначає, як слід позначати прозорість")), 0)
show_query(["значення", "колір", "прозорість"], known_item=known_target)
show_query(["перевищено", "час", "очікування"])
show_query(["не", "вдалося", "відкрити"])

## 14 · Одруківки: коли рятує `rapidfuzz`

Пошук по словах падає від однієї зайвої літери: «зннайдено» — це просто інша
колонка, і індекс про неї нічого не знає. Заміряємо, наскільки саме падає,
і чи допомагає нечіткий пошук по словнику.

In [ ]:
from rapidfuzz import process, distance, fuzz

LETTERS = 'абвгдеєжзиіїйклмнопрстуфхцчшщьюя'


def make_typo(word, rng):
    """Одна одруківка: переставити дві сусідні літери, викинути літеру або замінити."""
    kind = rng.integers(3)
    position = int(rng.integers(len(word)))
    if kind == 0 and len(word) > 2:
        j = min(position, len(word) - 2)
        return word[:j] + word[j + 1] + word[j] + word[j + 2:]
    if kind == 1 and len(word) > 3:
        return word[:position] + word[position + 1:]
    return word[:position] + LETTERS[int(rng.integers(len(LETTERS)))] + word[position + 1:]


def words_to_matrix(rows, vocabulary, n_columns):
    matrix_rows, matrix_columns = [], []
    for number, row in enumerate(rows):
        for word in row:
            if word in vocabulary:
                matrix_rows.append(number)
                matrix_columns.append(vocabulary[word])
    return sp.csr_matrix((np.ones(len(matrix_rows)), (matrix_rows, matrix_columns)),
                         shape=(len(rows), n_columns))


typo_table = {}
for level in (0, 1, 2, 3):
    broken_runs, fixed_runs = [], []
    for seed in SEEDS:
        texts, vectorizer, counts = SAMPLES[seed]
        vocabulary = vectorizer.vocabulary_
        vocabulary_words = list(vectorizer.get_feature_names_out())
        queries, targets = QUERIES[seed]
        clean = [[vocabulary_words[t] for t in queries.indices[queries.indptr[i]:queries.indptr[i + 1]]]
                 for i in range(queries.shape[0])]
        rng = np.random.default_rng(500 + seed)
        broken = []
        for row in clean:
            spoil = set(rng.permutation(len(row))[:level])
            broken.append([make_typo(w, rng) if k in spoil else w for k, w in enumerate(row)])
        flat = [word for row in broken for word in row]
        similarity = process.cdist(flat, vocabulary_words,
                                   scorer=distance.Levenshtein.normalized_similarity, workers=-1)
        nearest = similarity.argmax(axis=1)
        closeness = similarity.max(axis=1)
        repaired_flat = [vocabulary_words[j] if (w not in vocabulary and c >= 0.6) else w
                         for w, j, c in zip(flat, nearest, closeness)]
        repaired, cursor = [], 0
        for row in broken:
            repaired.append(repaired_flat[cursor:cursor + len(row)])
            cursor += len(row)
        broken_runs.append(mrr_of(bm25_scores(counts, words_to_matrix(broken, vocabulary, counts.shape[1])), targets)[0])
        fixed_runs.append(mrr_of(bm25_scores(counts, words_to_matrix(repaired, vocabulary, counts.shape[1])), targets)[0])
    typo_table[level] = (np.mean(broken_runs), np.mean(fixed_runs))
    print(f"зіпсовано {level} слова з 3: MRR без виправлення {np.mean(broken_runs):.4f} "
          f"(розкид {max(broken_runs) - min(broken_runs):.4f}), "
          f"після rapidfuzz {np.mean(fixed_runs):.4f} "
          f"(розкид {max(fixed_runs) - min(fixed_runs):.4f})")

Виправлення коштує часу, і вибір міри відстані коштує дуже по-різному.
Заміряємо дві: звичайну відстань Левенштейна і відстань Дамерау-Левенштейна,
яка вважає перестановку сусідніх літер **однією** правкою, а не двома.

In [ ]:
texts, vectorizer, counts = SAMPLES[0]
vocabulary_words = list(vectorizer.get_feature_names_out())
rng = np.random.default_rng(500)
sample_words = [make_typo(w, rng) for w in vocabulary_words[:900]]

for name, scorer in [("Левенштейн", distance.Levenshtein.normalized_similarity),
                     ("Дамерау-Левенштейн", distance.DamerauLevenshtein.normalized_similarity)]:
    start = time.perf_counter()
    matrix = process.cdist(sample_words[:300], vocabulary_words, scorer=scorer, workers=-1)
    elapsed = time.perf_counter() - start
    restored = np.mean([vocabulary_words[j] == w for j, w in
                        zip(matrix.argmax(axis=1), vocabulary_words[:300])])
    print(f"{name:<20} {elapsed:6.2f} с на 300 слів проти словника з "
          f"{len(vocabulary_words)}, вгадано слово точно {100 * restored:.1f} %")

## 15 · Де нечіткий пошук безсилий

Одруківка — це коли **той самий зміст записаний майже тими самими символами**.
Синонім — коли зміст той самий, а символи інші. Нечіткий пошук міряє символи,
тож друге він не бачить принципово. Перевіримо це числом.

Пари беремо ті самі, що тема 05: один англійський рядок, два різні українські
переклади, майже без спільних слів.

In [ ]:
NOISE = re.compile(r"[@<>]|https?://")


def paraphrase_pairs():
    """Пари українських перекладів того самого англійського рядка,
    у яких майже немає спільних слів."""
    by_source = collections.defaultdict(dict)
    for program, source, target in corpus:
        if source.strip() == 'translator-credits':
            continue
        by_source[source][' '.join(target.split())] = program
    found = []
    for source, variants in by_source.items():
        variant_texts = list(variants)
        for i in range(len(variant_texts)):
            for j in range(i + 1, len(variant_texts)):
                first, second = variant_texts[i], variant_texts[j]
                if NOISE.search(first) or NOISE.search(second):
                    continue
                a = set(tokenize(first.lower()))
                b = set(tokenize(second.lower()))
                if len(a) < 3 or len(b) < 3:
                    continue
                if len(a & b) / len(a | b) < 0.34:
                    found.append((first, second, len(a & b)))
    return found


pairs = paraphrase_pairs()
no_overlap = [p for p in pairs if p[2] == 0]
print(f"пар «те саме іншими словами»: {len(pairs)}, "
      f"з них без жодного спільного слова: {len(no_overlap)}")
for first, second, _ in no_overlap[:3]:
    print(f"   A: {first}")
    print(f"   B: {second}")

In [ ]:
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    same_meaning, same_text_with_typos, unrelated = [], [], []
    for first, second, _ in no_overlap:
        same_meaning.append(fuzz.token_sort_ratio(first.lower(), second.lower()) / 100)
        spoiled = ' '.join(make_typo(w, rng) if len(w) > 3 else w for w in first.split())
        same_text_with_typos.append(fuzz.token_sort_ratio(first.lower(), spoiled.lower()) / 100)
    sample = sample_documents(seed, n=2000)
    picks = rng.choice(len(sample), (len(no_overlap), 2))
    for i, j in picks:
        unrelated.append(fuzz.token_sort_ratio(sample[i].lower(), sample[j].lower()) / 100)
    print(f"зерно {seed}: схожість рядків — "
          f"той самий текст з одруківками {np.mean(same_text_with_typos):.4f}, "
          f"той самий зміст іншими словами {np.mean(same_meaning):.4f}, "
          f"випадкові пари {np.mean(unrelated):.4f}")

## 16 · Чи лікує BM25 сліпоту до синонімів

Головне питання розділу, і воно перевіряється прямо. Кладемо другу половину
кожної пари в колекцію, запитуємо словами першої половини й дивимось, чи
знайдеться потрібне.

Контроль обовʼязковий: той самий замір, але запит складено зі слів **самого
документа-мішені**. Якщо контроль високий, а основний замір низький — справа
не в задачі й не в колекції, а саме в тому, що слова різні.

In [ ]:
def paraphrase_search(scorer):
    quality, control, recall = [], [], []
    for seed in SEEDS:
        base = sample_documents(seed)
        collection = base + [second for _, second, _ in pairs]
        target_ids = np.arange(len(base), len(collection))
        local_vectorizer, local_counts = count_matrix(collection)
        local_vocabulary = local_vectorizer.vocabulary_

        def query_matrix(get_text):
            rows, columns = [], []
            for number, pair in enumerate(pairs):
                for word in set(tokenize(get_text(pair).lower())):
                    if word in local_vocabulary:
                        rows.append(number)
                        columns.append(local_vocabulary[word])
            return sp.csr_matrix((np.ones(len(rows)), (rows, columns)),
                                 shape=(len(pairs), local_counts.shape[1]))

        scores = scorer(local_counts, query_matrix(lambda p: p[0]))
        value, places = mrr_of(scores, target_ids)
        quality.append(value)
        recall.append(float(np.mean(places <= 10)))
        control_scores = scorer(local_counts, query_matrix(lambda p: p[1]))
        control.append(mrr_of(control_scores, target_ids)[0])
    return quality, control, recall


for name, scorer in [("TF-IDF", lambda c, q: tfidf_scores(c, q)),
                     ("BM25", lambda c, q: bm25_scores(c, q, 1.5, 0.75))]:
    quality, control, recall = paraphrase_search(scorer)
    print(f"{name:<8} запит іншими словами: MRR {np.mean(quality):.4f} "
          f"(розкид {max(quality) - min(quality):.4f}), recall@10 {np.mean(recall):.4f}")
    print(f"{'':<8} контроль, запит словами самого документа: MRR {np.mean(control):.4f}")

## 17 · Підсумкова таблиця теми

Усе, що заміряно, в одному місці — щоб було видно, що чого варте.

In [ ]:
summary = [
    ("індекс проти перебору", f"{scan_time / index_time:.1f}x на 20 000 документів"),
    ("торкаємось документів на запит", "від 0.13 % (рідкісні слова) до 30 % (часті)"),
    ("TF-IDF косинус, MRR", f"{np.mean(main_table['TF-IDF косинус']):.4f}"),
    ("BM25 k1=1.5 b=0.75, MRR", f"{np.mean(main_table['BM25 k1=1.5 b=0.75']):.4f}"),
    ("виграш BM25", f"{np.mean(gains):+.4f} при розкиді {max(gains) - min(gains):.4f}"),
    ("ціна вимкненого насичення", f"{np.mean(ablation_runs['6 BM25 без насичення, k1=1000']) - full:+.4f}"),
    ("найкраща клітинка сітки", f"k1={best_cell[0]} b={best_cell[1]}, MRR {best_value:.4f}"),
    ("одруківка в кожному слові", f"MRR {typo_table[3][0]:.4f}, після rapidfuzz {typo_table[3][1]:.4f}"),
]
width = max(len(name) for name, _ in summary)
for name, value in summary:
    print(f"{name:<{width}} : {value}")

## 18 · Дані для фігур лекції

Числа, які лекція показує в інтерактивах, беруться звідси. Друкуємо їх окремо,
щоб при зміні корпусу було видно, що саме поїхало.

In [ ]:
figure_data = {
    "index": [(" ".join(q), None) for q in DEMO_QUERIES],
    "saturation": [(tf, round(value, 4)) for tf, value in saturation_table],
    "length": [(b, round(quality, 4), round(top, 2)) for b, quality, top in length_table],
    "compare": {name: [round(v, 4) for v in runs] for name, runs in ablation_runs.items()},
    "grid": {f"k1={k1} b={b}": round(grid[(k1, b)][0], 4) for k1 in K1_GRID for b in B_GRID},
    "typos": {level: (round(a, 4), round(c, 4)) for level, (a, c) in typo_table.items()},
}
for section, payload in figure_data.items():
    print(f"── {section}")
    print("   ", payload)

## 19 · Завдання

### 🟢 Рівень 1

Додай до сітки `k1` × `b` ще один стовпчик — `b = 0.15` — і подивись, чи
залишиться найкраща клітинка на місці. **Зроблено, якщо** ти назвав нову
найкращу пару `(k1, b)` і сказав, чи більша її перевага за розкид по зернах.

### 🟡 Рівень 2

Побудуй інвертований індекс **по лемах** замість словоформ (`pymorphy3` із
теми 03) і перевір, чи виросте MRR на запитах типу `mixed`. **Зроблено, якщо**
є число до й після на трьох зернах і висновок, більша різниця за розкид чи ні.

### 🔴 Рівень 3

Реалізуй ранній вихід з індексу: обходь списки документів у порядку
зростання `df`, тримай купу з десяти найкращих і припиняй обхід списку, коли
навіть максимально можлива добавка решти слів не піднімає документ у топ-10
(це відомий алгоритм WAND). **Зроблено, якщо** видача збігається з повним
підрахунком символ у символ, а кількість оброблених пар «слово-документ»
менша хоча б удвічі.